In [ ]:
# Please install OpenAI SDK first: `pip3 install openai`
import os
from openai import OpenAI
import json
import pandas as pd

from sklearn.model_selection import train_test_split

In [ ]:
DEEPSEEK_API_KEY='...'
client = OpenAI(api_key=DEEPSEEK_API_KEY, base_url="https://api.deepseek.com")

response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "system", "content": "You are a helpful assistant"},
        {"role": "user", "content": "Hello"},
    ],
    stream=False
)

print(response.choices[0].message.content)

In [ ]:
msg_text = "Just tried the new iPhone feature and it blew my mind 🤯🔥"

system_prompt = """
You are a tweet virality scoring model.
Your job: read the tweet text and return a virality score.

Virality score rules:
0 = low viral (ordinary, niche, no hook, low engagement potential)
1 = medium viral (interesting, relatable, decent hook, moderate engagement potential)
2 = high viral (strong hook, highly shareable, controversial, emotional, breaking news, meme-worthy, strong CTA)

Return ONLY valid JSON with this exact schema:
{
  "virality_score": 0|1|2
}
No extra keys. No explanation. No markdown.
"""



In [ ]:
response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": msg_text},
    ],
    temperature=0,          # more consistent scoring
    max_tokens=20,
    stream=False
)

raw = response.choices[0].message.content.strip()

# Optional: validate JSON output
result = json.loads(raw)
print(result)

In [ ]:
def get_response(msg_text,system_prompt):
    response = client.chat.completions.create(
        model="deepseek-chat",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": msg_text},
        ],
        temperature=0,          # more consistent scoring
        max_tokens=20,
        stream=False
    )

    raw = response.choices[0].message.content.strip()
    return raw

    # Optional: validate JSON output
    
    

In [ ]:
df=pd.read_csv('train_df.csv')

In [ ]:
# split into train and test

df_train, df_test,  = train_test_split(
    df, test_size=0.93,
    stratify=df['ViralityLabel'],
      random_state=42)

print(df_train.shape,df_test.shape)
df_train.to_csv('train_df_train.csv',index=False)
df_test.to_csv('train_df_test.csv',index=False)

In [ ]:
df=pd.read_csv('train_df_train.csv')

In [ ]:
res=[]
for idx,row in df.iterrows():
    print(row['Text'])
    msg_text=row['Text']
    raw=get_response(msg_text,system_prompt)
    result = json.loads(raw)
    print(raw,result)        
    res.append({
        'ID':row['ID'],
        'label':int(result['virality_score'])
    })
    res_df=pd.DataFrame(res)
    res_df.to_csv('Out/trial0/deepseek-chat_train_Text_labels.csv',index=False)   

# also save the system prompt
with open('Out/trial0/sys_prompt.txt','w') as f:
    f.write(system_prompt)

In [ ]:
result

## Can we try openrouter API

In [ ]:
import requests
import json

openrouter_api_key='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'



In [ ]:
response = requests.post(
  url="https://openrouter.ai/api/v1/chat/completions",
  headers={
    "Authorization": f"Bearer {openrouter_api_key}",    
  },
  data=json.dumps({
    "model": "openai/gpt-5.2", # Optional
    "messages": [
      {
        "role": "user",
        "content": "What is the meaning of life?"
      }
    ]
  })
)

result = response.json()
print(result['choices'][0]['message']['content'])


In [ ]:
system_prompt = """
You are a tweet virality scoring model.
Your job: read the tweet text and return a virality score.

Virality score rules:
0 = low viral (ordinary, niche, no hook, low engagement potential)
1 = medium viral (interesting, relatable, decent hook, moderate engagement potential)
2 = high viral (strong hook, highly shareable, controversial, emotional, breaking news, meme-worthy, strong CTA)

Return ONLY valid JSON with this exact schema:
{
  "virality_score": 0|1|2
}
No extra keys. No explanation. No markdown.
"""

user_tweet = "OpenAI just released a new model that beats humans at reasoning."

response = requests.post(
    url="https://openrouter.ai/api/v1/chat/completions",
    headers={
        "Authorization": f"Bearer {openrouter_api_key}",
        "Content-Type": "application/json",
    },
    json={
        "model": "openai/gpt-5.2",
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_tweet},
        ],
        "response_format": {"type": "json_object"},
        "temperature": 0,
    },
)

result = response.json()
content = result["choices"][0]["message"]["content"]

# content is a JSON string → parse it
parsed = json.loads(content)


print(parsed["virality_score"])

### Next we try Gemini

In [4]:
import requests
import json


openrouter_api_key='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'



max_tokens=100
msg_text="OpenAI just dropped a new model."
system_prompt = """
You are a tweet virality scoring model.
Your job: read the tweet text and return a virality score.

Virality score rules:
0 = low viral (ordinary, niche, no hook, low engagement potential)
1 = medium viral (interesting, relatable, decent hook, moderate engagement potential)
2 = high viral (strong hook, highly shareable, controversial, emotional, breaking news, meme-worthy, strong CTA)

Return ONLY valid JSON with this exact schema:
{
  "virality_score": 0|1|2
}
No extra keys. No explanation. No markdown.
"""



In [5]:
response = requests.post(
        url="https://openrouter.ai/api/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {openrouter_api_key}",
            "Content-Type": "application/json",
        },
        json={
            "model": "google/gemini-2.5-pro-preview",  # or gemini-1.5-flash
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": msg_text},
            ],
            "response_format": {"type": "json_object"},  # ✅ force JSON
            "temperature": 0,
            "max_tokens": max_tokens,
        },
    )

In [6]:
response.__dict__

{'_content': b'\n         \n\n         \n\n         \n\n         \n\n         \n\n         \n\n         \n{"id":"gen-1770697784-OOiIMt5YyU56EJndtyJk","provider":"Google","model":"google/gemini-2.5-pro","object":"chat.completion","created":1770697784,"choices":[{"logprobs":null,"finish_reason":"length","native_finish_reason":"MAX_TOKENS","index":0,"message":{"role":"assistant","content":"Here is","refusal":null,"reasoning":"**Understanding the User\'s Goal**\\n\\nI\'m focused on grasping the essence of the user\'s request. It seems I must embody a \\"tweet virality scoring model\\" and provide a score. My task includes reading a tweet and producing a JSON with a virality score, specifically 0, 1, or 2. This seems straightforward.\\n\\n\\n**Evaluating Tweet Content**\\n\\nI\'m now delving into the scoring criteria. A score of \'0\' means a tweet is low-viral: ordinary, niche, or hookless. A score of \'2\' indicates high virality, and must have a strong hook, be very popular, or use trend

In [7]:
response_json=json.loads(response.content)

In [8]:
response_json

{'id': 'gen-1770697784-OOiIMt5YyU56EJndtyJk',
 'provider': 'Google',
 'model': 'google/gemini-2.5-pro',
 'object': 'chat.completion',
 'created': 1770697784,
 'choices': [{'logprobs': None,
   'finish_reason': 'length',
   'native_finish_reason': 'MAX_TOKENS',
   'index': 0,
   'message': {'role': 'assistant',
    'content': 'Here is',
    'refusal': None,
    'reasoning': '**Understanding the User\'s Goal**\n\nI\'m focused on grasping the essence of the user\'s request. It seems I must embody a "tweet virality scoring model" and provide a score. My task includes reading a tweet and producing a JSON with a virality score, specifically 0, 1, or 2. This seems straightforward.\n\n\n**Evaluating Tweet Content**\n\nI\'m now delving into the scoring criteria. A score of \'0\' means a tweet is low-viral: ordinary, niche, or hookless. A score of \'2\' indicates high virality, and must have a strong hook, be very popular, or use trending hashtags. A \'1\' suggests a moderate degree of virality.